In [ ]:
import pandas as pd

In [ ]:
sales_df = pd.read_csv('test/sales_raw.csv')

In [ ]:
discounts_df = pd.read_csv('test/discounts_raw.csv')

# EDA

## ???????? ???? ?????? discounts_df ? ???????? ? datetype

In [ ]:
discounts_df.head()

In [ ]:
discounts_df.info()

In [ ]:
discounts_df['date_start'] = pd.to_datetime(discounts_df['date_start'])
discounts_df['date_end'] = pd.to_datetime(discounts_df['date_end'])

In [ ]:
discounts_df.info()

## ?????????/?????????? ??????? ?????? ????? discounts_df

In [ ]:
print(f'Missing values: {discounts_df.isnull().sum().values.sum()}')
discounts_df.isnull().sum()

### ????????? ??????? ? moda[0]

In [ ]:
discounts_df= discounts_df.apply(lambda x: x.fillna(x.value_counts().index[0]))

In [ ]:
discounts_df.isnull().sum()

In [ ]:
discounts_df.groupby('promo_typeCode').count()

## ???????? ???? ?????? sales_df ? ???????? ? datetype

In [ ]:
print('sales shape: ', sales_df.shape)
sales_df.head()

In [ ]:
sales_df.info()

In [ ]:
sales_df['date'] = pd.to_datetime(sales_df['date'])

In [ ]:
print(f'Missing values: {sales_df.isnull().sum().values.sum()}')
sales_df.isnull().sum()

## ?????? ??????? ??????? ??????????? ? ???????? ? ??????? 2023

In [ ]:
dec_sales_items = sales_df['item_id'][(sales_df['date']<'2024-01-01')&(sales_df['date']>'2023-11-30')].reset_index()
dec_sales_items = dec_sales_items['item_id']
dec_sales_items_list = pd.unique(dec_sales_items)

In [ ]:
dec_sales_items_list

## ???????

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
sales_groupby_df = sales_df[(sales_df['item_id'].isin(dec_sales_items_list))]
sales_groupby_df = sales_groupby_df.groupby('date').sum().reset_index()
sales_groupby_df['date'] = pd.to_datetime(sales_groupby_df['date'])
sales_groupby_df.drop(columns=['item_id'], inplace=True)

In [ ]:
sales_groupby_df = sales_groupby_df[(sales_groupby_df['date']<'2024-01-01')&(sales_groupby_df['date']>'2023-11-30')]
sales_groupby_df.head(5)
# sales_groupby_df.info()

In [ ]:
sns.set_theme()
sns.relplot(data = sales_groupby_df, kind = 'line', x = 'date', y = 'qnty',
    facet_kws=dict(sharex=False))

# ?????????? ?????? ??? ????????

? ??? ???? 3 df ? 1 ???? ? ???????? ????? ???????? 
?? ????? ?? ??????????
1. discounts_df_1
2. sales_df
3. dec_sales_items_list - items_id ??????? ??????? ??????????? ? ??????? 2023

## ???????

In [ ]:
discounts_df.shape, sales_df.shape

In [ ]:
def format_df(salesdf, discountsdf):
    r_dataset = pd.DataFrame(columns=['date', 'item_id', 'promo_typeCode', 'sale_price', 'sold'])
    for index, row in salesdf.iterrows():
        x = discountsdf[(discountsdf['item_id'] == row['item_id']) & (row['date']>discountsdf['date_start']) & (row['date']<discountsdf['date_end'])]
        if len(x)==1:
            r_dataset.loc[len(r_dataset.index)] = [row['date'], row['item_id'], x.iloc[0]['promo_typeCode'], x.iloc[0]['sale_price_time_promo'], row['qnty']]
        elif x.empty:
            y = discountsdf[(discountsdf['item_id'] == row['item_id']) & (row['date']>discountsdf['date_start']) & (discountsdf['date_start'] <= discountsdf['date_start'].min())]
            if not y.empty:
                r_dataset.loc[len(r_dataset.index)] = [row['date'], row['item_id'], 0.0, y.iloc[0]['sale_price_before_promo'], row['qnty']]
        else:
            x = x[x['sale_price_time_promo'] == x['sale_price_time_promo'].min()]
            r_dataset.loc[len(r_dataset.index)] = [row['date'], row['item_id'], x.iloc[0]['promo_typeCode'], x.iloc[0]['sale_price_time_promo'], row['qnty']]
    data_dummyes = pd.get_dummies(r_dataset['promo_typeCode'], prefix='promo', drop_first=True, dtype=float)
    r_dataset = pd.concat([r_dataset, data_dummyes], axis=1)
    r_dataset = r_dataset.drop(['promo_typeCode'], axis=1)
    return r_dataset

In [ ]:
def format_predict_df(salesdf, item_list, discountsdf):
    r_dataset = pd.DataFrame(columns=['date', 'item_id', 'promo_typeCode', 'sale_price'])
    for index, row in salesdf.iterrows():
        for item_id in item_list:
            x = discountsdf[(discountsdf['item_id'] == item_id) & (row['date']>discountsdf['date_start']) & (row['date']<discountsdf['date_end'])]
            if len(x)==1:
                r_dataset.loc[len(r_dataset.index)] = [row['date'], item_id, x.iloc[0]['promo_typeCode'], x.iloc[0]['sale_price_time_promo']]
            elif x.empty:
                y = discountsdf[(discountsdf['item_id'] == item_id) & (row['date']>discountsdf['date_start']) & (discountsdf['date_start'] <= discountsdf['date_start'].min())]
                if not y.empty:
                    r_dataset.loc[len(r_dataset.index)] = [row['date'], item_id, 0.0, y.iloc[0]['sale_price_before_promo']]
            else:
                x = x[x['sale_price_time_promo'] == x['sale_price_time_promo'].min()]
                r_dataset.loc[len(r_dataset.index)] = [row['date'], item_id, x.iloc[0]['promo_typeCode'], x.iloc[0]['sale_price_time_promo']]

    data_dummyes = pd.get_dummies(r_dataset['promo_typeCode'], prefix='promo', drop_first=True, dtype=float)
    r_dataset = pd.concat([r_dataset, data_dummyes], axis=1)
    r_dataset = r_dataset.drop(['promo_typeCode'], axis=1)
    r_dataset['promo_1.0']=0.0
    r_dataset['promo_2.0']=0.0
    r_dataset['promo_7.0']=0.0
    r_dataset['promo_10.0']=0.0
    r_dataset['promo_16.0']=0.0
    r_dataset = r_dataset[['date', 'item_id', 'sale_price', 'promo_1.0', 'promo_2.0','promo_5.0','promo_6.0','promo_7.0','promo_8.0', 'promo_10.0', 'promo_16.0','promo_44.0']]
    return r_dataset

In [ ]:
pred_start_date = pd.to_datetime('2024-01-01')
pred_end_date = pd.to_datetime('2024-01-31')
pred_date_range = pd.date_range(start=pred_start_date, end=pred_end_date)
pred_df_dates = pd.DataFrame()
pred_df_dates['date'] = pred_date_range
pred_df_dates.tail()

## ?????? ??? ????????

In [ ]:
# ?????
train_df = format_df(sales_df, discounts_df)

In [ ]:
pred_df = format_predict_df(pred_df_dates, dec_sales_items_list, discounts_df)

### dummies for item_id

In [ ]:
item_id_dummyes = pd.get_dummies(train_df['item_id'], prefix='item_id', drop_first=True, dtype=float)
item_id_dummyes.head()

In [ ]:
train_df_2 = pd.concat([train_df.drop(['item_id'], axis = 1), item_id_dummyes], axis=1)
train_df_2.head()

In [ ]:
train_df_2 = train_df_2[[c for c in train_df_2 if c not in ['sold']] + ['sold']]
train_df_2.head()

In [ ]:
train_df_2.iloc[:,:-1]

# ????????

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsRegressor

import numpy as np
from math import sqrt, ceil
import datetime as dt

In [ ]:
train_df_lr = train_df.copy()
pred_df_lr = pred_df.copy()

In [ ]:
train_df_lr['date'] = pd.to_numeric(train_df_lr['date'])
pred_df_lr['date'] = pd.to_numeric(pred_df_lr['date'])

In [ ]:
# X_train, X_test, y_train, y_test = train_test_split(train_df_2.iloc[:,:-1], train_df_2.iloc[:,-1], test_size=0.25, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(train_df_lr.drop(['sold'], axis=1), train_df_lr['sold'], test_size=0.25, random_state=42)

## ???????? ?????????

In [ ]:
linreg_model = LinearRegression()

In [ ]:
linreg_model.fit(X_train, y_train)

In [ ]:
y_pred = linreg_model.predict(X_test)
rms = sqrt(mean_squared_error(y_pred, y_test))
print(f'RMSE value: {rms}')

In [ ]:
lr_jan_pred = linreg_model.predict(pred_df_lr)

## ARIMA

In [ ]:
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA

In [ ]:
result = adfuller(train_df_lr['sold'])
print('ADF ??????????:', result[0])
print('p-????????:', result[1])
print('??????????? ????????:')
for key, value in result[4].items():
    print(f'  {key}: {value}')

In [ ]:
model = ARIMA(train_df_lr['sold'], order=(1, 1, 1))
model_fit = model.fit()

# ????? ?????????? ??????
print(model_fit.summary())

In [ ]:
# ??????? ?? ?????? ????????? ??????
forecast = model_fit.forecast(steps=12)

# ???????????? MSE ? rmse
mse = mean_squared_error(train_df_lr['sold'][-12:], forecast)
rmse = sqrt(mse)

print(f'MSE: {mse}')
print(f'rmse: {rmse}')

## k near

In [ ]:
shape = train_df.shape[0]
ttrain_set=train_df.iloc[:ceil(shape*0.75)]
ttest_set=train_df.iloc[ceil(shape*0.75):]
x_ttrain = ttrain_set.drop('sold', axis=1)
x_ttest = ttest_set.drop('sold', axis=1)
x_ttrain['date'] = ttrain_set['date'].map(dt.datetime.toordinal)
y_ttrain = ttrain_set['sold']
x_ttest['date'] = ttest_set['date'].map(dt.datetime.toordinal)
y_ttest = ttest_set['sold']

In [ ]:
params = {'n_neighbors': [2,3,4,5,6,7,8,9]}
knn = KNeighborsRegressor()
model = GridSearchCV(knn, params, cv=5)


In [ ]:
model.fit(x_ttrain, y_ttrain)

In [ ]:
knn_preds = model.predict(x_ttest)

In [ ]:
rms = sqrt(mean_squared_error(knn_preds, y_ttest))
print(f'RMSE value: {rms}')

In [ ]:
knn_jan_data = format_predict_df(pred_df_dates, dec_sales_items_list, discounts_df)
knn_jan_data['date'] = knn_jan_data['date'].map(dt.datetime.toordinal)

In [ ]:
knn_result = model.predict(knn_jan_data)

# ?????????

? ?????????? knn ??? ????? ?????? ??????
1. knn rmse = 6.94
2. lr rmse  = 10.88
3. arima rmse = 18

In [ ]:
jan2024_df = format_predict_df(pred_df_dates, dec_sales_items_list, discounts_df)

In [ ]:
knn_result = pd.DataFrame(knn_result, columns=['knn_result'])

In [ ]:
lr_result = pd.DataFrame(lr_jan_pred, columns=['lr_result'])

In [ ]:
jan2024_df_knn = pd.concat([jan2024_df[['date', 'item_id']],knn_result], axis =1 )

In [ ]:
jan2024_df_lr = pd.concat([jan2024_df[['date', 'item_id']], lr_result], axis =1 )


In [ ]:
jan2024_df_knn.to_csv('jan2024_df_knn.csv', sep=';')
jan2024_df_knn

In [ ]:
jan2024_df_lr